# Module: Transition from Single Population Inference to Comparative Inference

## Overview

In this notebook, we will transition from analyzing single populations to comparing multiple populations. This is a fundamental shift in statistical thinking that powers real-world decision making.

## 1. From Single Population to Comparative Inference

### Single Population Inference (Previous Modules)

Previously, we focused on questions like:
- Estimating a single population mean: mu
- Testing a single proportion: H0: p = 0.5
- Constructing confidence intervals for one parameter

### Comparative Inference (This Module)

Now we ask comparative questions:
- Is mu1 - mu2 = 0?
- Is p1 = p2?
- Are multiple population means equal?
- Are categorical variables associated?

**Key Insight:** Uncertainty now comes from multiple samples simultaneously.

## 2. Why Comparative Inference Matters

Real-world decisions are fundamentally comparative:

| Domain | Comparative Question |
|--------|---------------------|
| Marketing | Which campaign generates higher conversion? |
| Medicine | Which treatment improves recovery more? |
| Manufacturing | Which process has lower defect rates? |
| Finance | Which portfolio produces higher returns? |
| Education | Which teaching method improves scores more? |

A single isolated number is rarely sufficient for decision making.

## 3. Learning Objective 1: Comparing Two Population Means

### Two-Sample t-Test

When comparing two independent populations, we test:

H0: mu1 = mu2 (or mu1 - mu2 = 0)

HA: mu1 != mu2 (two-tailed)

Or one-tailed alternatives: mu1 < mu2 or mu1 > mu2

### Assumptions:
1. Independence within and between samples
2. Normality (or large sample sizes)
3. Equal variances (or use Welch's t-test)

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)

# Generate sample data for two groups
group_A = np.random.normal(loc=100, scale=15, size=50)
group_B = np.random.normal(loc=108, scale=15, size=50)

# Display basic statistics
print(f"Group A - Mean: {group_A.mean():.2f}, Std: {group_A.std():.2f}")
print(f"Group B - Mean: {group_B.mean():.2f}, Std: {group_B.std():.2f}")
print(f"Difference in means: {group_B.mean() - group_A.mean():.2f}")

In [ ]:
# Perform two-sample t-test
t_stat, p_value = stats.ttest_ind(group_A, group_B)

print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")

# Decision at alpha = 0.05
alpha = 0.05
if p_value < alpha:
    print("Reject H0: There is a significant difference between the two population means")
else:
    print("Fail to reject H0: No significant difference detected")

# Welch's t-test (does not assume equal variances)
t_stat_welch, p_value_welch = stats.ttest_ind(group_A, group_B, equal_var=False)
print(f"\nWelch's t-test - p-value: {p_value_welch:.4f}")

In [ ]:
# Visualize the two distributions
plt.figure(figsize=(10, 6))
plt.hist(group_A, alpha=0.5, label='Group A', bins=15, edgecolor='black')
plt.hist(group_B, alpha=0.5, label='Group B', bins=15, edgecolor='black')
plt.axvline(group_A.mean(), color='blue', linestyle='dashed', linewidth=2, label=f'Mean A: {group_A.mean():.2f}')
plt.axvline(group_B.mean(), color='orange', linestyle='dashed', linewidth=2, label=f'Mean B: {group_B.mean():.2f}')
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.title('Distribution Comparison: Group A vs Group B')
plt.legend()
plt.show()

## 4. ANOVA: Comparing More Than Two Population Means

When comparing k > 2 populations, performing multiple t-tests increases Type I error. ANOVA (Analysis of Variance) solves this problem.

**Null Hypothesis:** H0: mu1 = mu2 = mu3 = ... = muk

**Alternative Hypothesis:** At least one mui differs

### Logic of ANOVA:

F = Between-group variability / Within-group variability

Large F -> Evidence against H0

In [ ]:
# Generate data for 4 groups
groups = {
    'Campaign A': np.random.normal(loc=100, scale=12, size=40),
    'Campaign B': np.random.normal(loc=105, scale=12, size=40),
    'Campaign C': np.random.normal(loc=98, scale=12, size=40),
    'Campaign D': np.random.normal(loc=110, scale=12, size=40)
}

# Create DataFrame
df_anova = pd.DataFrame([(campaign, value) for campaign, values in groups.items() for value in values],
                        columns=['Campaign', 'Sales'])

# Display means
means = df_anova.groupby('Campaign')['Sales'].mean()
print("Group Means:")
print(means)
print(f"\nOverall Mean: {df_anova['Sales'].mean():.2f}")

In [ ]:
# Perform One-Way ANOVA
group_data = [groups[key] for key in groups.keys()]
f_stat, p_value_anova = stats.f_oneway(*group_data)

print(f"F-statistic: {f_stat:.4f}")
print(f"p-value: {p_value_anova:.4f}")

if p_value_anova < 0.05:
    print("Reject H0: At least one campaign mean differs significantly")
else:
    print("Fail to reject H0: No significant difference among campaign means")

In [ ]:
# Visualize ANOVA results
plt.figure(figsize=(10, 6))
sns.boxplot(data=df_anova, x='Campaign', y='Sales')
plt.title('Sales Distribution by Marketing Campaign')
plt.xlabel('Campaign')
plt.ylabel('Sales')
plt.show()

In [ ]:
# Post-hoc analysis (Tukey's HSD)
from statsmodels.stats.multicomp import pairwise_tukeyhsd

tukey_result = pairwise_tukeyhsd(df_anova['Sales'], df_anova['Campaign'], alpha=0.05)
print("Tukey HSD Post-hoc Test Results:")
print(tukey_result)

## 5. Learning Objective 2: Comparing Population Variances

Sometimes variability matters more than the mean. Examples:
- Manufacturing consistency
- Investment volatility
- Quality control

### F-Test for Equality of Variances

H0: sigma1^2 = sigma2^2

HA: sigma1^2 != sigma2^2

Test statistic: F = s1^2 / s2^2 (where s1^2 >= s2^2)

In [ ]:
# Generate data with different variances
process_1 = np.random.normal(loc=100, scale=5, size=30)
process_2 = np.random.normal(loc=100, scale=15, size=30)

print(f"Process 1 - Variance: {process_1.var():.2f}")
print(f"Process 2 - Variance: {process_2.var():.2f}")
print(f"Variance Ratio (larger/smaller): {max(process_1.var(), process_2.var()) / min(process_1.var(), process_2.var()):.2f}")

# F-test for variances
f_stat_var = process_2.var() / process_1.var()
df1 = len(process_2) - 1
df2 = len(process_1) - 1
p_value_var = 2 * min(1 - stats.f.cdf(f_stat_var, df1, df2), stats.f.cdf(f_stat_var, df1, df2))

print(f"\nF-statistic: {f_stat_var:.4f}")
print(f"p-value: {p_value_var:.4f}")

# Levene's test (more robust)
levene_stat, levene_p = stats.levene(process_1, process_2)
print(f"Levene's Test - p-value: {levene_p:.4f}")

In [ ]:
# Visualize variance difference
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.boxplot([process_1, process_2], labels=['Process 1', 'Process 2'])
ax1.set_title('Boxplot: Process Comparison')
ax1.set_ylabel('Output')

ax2.hist(process_1, alpha=0.5, label='Process 1 (Low Variance)', bins=15, edgecolor='black')
ax2.hist(process_2, alpha=0.5, label='Process 2 (High Variance)', bins=15, edgecolor='black')
ax2.set_title('Histogram: Process Comparison')
ax2.set_xlabel('Output')
ax2.set_ylabel('Frequency')
ax2.legend()

plt.tight_layout()
plt.show()

## 6. Learning Objective 3: Comparing Population Proportions

For binary/categorical outcomes (success/failure, click/no click, churn/no churn):

### Two-Proportion Z-Test

H0: p1 = p2

HA: p1 != p2

This is the foundation of A/B testing in digital analytics!

In [ ]:
# A/B Testing Example
conversions_A = 120
visitors_A = 1000
conversions_B = 150
visitors_B = 1000

p1 = conversions_A / visitors_A
p2 = conversions_B / visitors_B

print(f"Website A Conversion Rate: {p1*100:.2f}%")
print(f"Website B Conversion Rate: {p2*100:.2f}%")
print(f"Difference: {(p2 - p1)*100:.2f} percentage points")

# Pooled proportion
p_pooled = (conversions_A + conversions_B) / (visitors_A + visitors_B)

# Standard error
se = np.sqrt(p_pooled * (1 - p_pooled) * (1/visitors_A + 1/visitors_B))

# Z-statistic
z_stat = (p2 - p1) / se

# P-value (two-tailed)
p_value_prop = 2 * (1 - stats.norm.cdf(abs(z_stat)))

print(f"\nZ-statistic: {z_stat:.4f}")
print(f"P-value: {p_value_prop:.4f}")

# Using statsmodels
from statsmodels.stats.proportion import proportions_ztest

count = np.array([conversions_A, conversions_B])
nobs = np.array([visitors_A, visitors_B])
z_stat_sm, p_value_sm = proportions_ztest(count, nobs, alternative='two-sided')

print(f"\nStatsmodels Z-statistic: {z_stat_sm:.4f}")
print(f"Statsmodels P-value: {p_value_sm:.4f}")

# Confidence interval
diff = p2 - p1
margin_of_error = 1.96 * se
ci_lower = diff - margin_of_error
ci_upper = diff + margin_of_error

print(f"\n95% CI for difference in proportions: [{ci_lower:.4f}, {ci_upper:.4f}]")
print(f"Converted to percentages: [{ci_lower*100:.2f}%, {ci_upper*100:.2f}%]")

In [ ]:
# Visualize A/B test results
categories = ['Website A', 'Website B']
conversion_rates = [p1*100, p2*100]
errors = [1.96 * np.sqrt(p1*(1-p1)/visitors_A)*100, 1.96 * np.sqrt(p2*(1-p2)/visitors_B)*100]

plt.figure(figsize=(8, 6))
plt.bar(categories, conversion_rates, yerr=errors, capsize=10, color=['blue', 'green'], alpha=0.7)
plt.ylabel('Conversion Rate (%)')
plt.title('A/B Test: Conversion Rate Comparison with 95% CI')
plt.ylim(0, max(conversion_rates) + 5)
plt.grid(axis='y', alpha=0.3)
plt.show()

## 7. Chi-Square Tests for Categorical Data

### 7.1 Chi-Square Goodness-of-Fit Test

Tests if observed frequencies match expected frequencies.

H0: The data follows the claimed distribution

In [ ]:
# Example: Is a die fair?
observed = np.array([8, 12, 9, 11, 10, 10])
expected = np.array([10, 10, 10, 10, 10, 10])

# Calculate chi-square statistic
chi_square_gof = np.sum((observed - expected)**2 / expected)
df_gof = len(observed) - 1
p_value_gof = 1 - stats.chi2.cdf(chi_square_gof, df_gof)

print(f"Chi-square statistic: {chi_square_gof:.4f}")
print(f"Degrees of freedom: {df_gof}")
print(f"P-value: {p_value_gof:.4f}")

# Using scipy
chi2_stat, p_val = stats.chisquare(observed, expected)
print(f"\nScipy Chi-square: {chi2_stat:.4f}, p-value: {p_val:.4f}")

if p_val > 0.05:
    print("Fail to reject H0: The die appears to be fair")
else:
    print("Reject H0: The die is not fair")

In [ ]:
# Visualize goodness-of-fit
faces = ['1', '2', '3', '4', '5', '6']
x = np.arange(len(faces))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, observed, width, label='Observed', color='blue', alpha=0.7)
bars2 = ax.bar(x + width/2, expected, width, label='Expected', color='orange', alpha=0.7)

ax.set_xlabel('Die Face')
ax.set_ylabel('Frequency')
ax.set_title('Chi-Square Goodness-of-Fit: Die Fairness Test')
ax.set_xticks(x)
ax.set_xticklabels(faces)
ax.legend()
plt.show()

### 7.2 Chi-Square Test of Independence

Tests whether two categorical variables are associated.

H0: The variables are independent

HA: The variables are associated

In [ ]:
# Example: Is purchase behavior associated with gender?
data = np.array([[30, 20], [45, 5]])

contingency_df = pd.DataFrame(data, 
                              index=['Male', 'Female'],
                              columns=['Purchase', 'No Purchase'])
print("Contingency Table:")
print(contingency_df)

# Chi-square test of independence
chi2, p_value_ind, dof, expected_table = stats.chi2_contingency(data)

print(f"\nChi-square statistic: {chi2:.4f}")
print(f"Degrees of freedom: {dof}")
print(f"P-value: {p_value_ind:.4f}")

print("\nExpected frequencies under independence:")
expected_df = pd.DataFrame(expected_table,
                           index=['Male', 'Female'],
                           columns=['Purchase', 'No Purchase'])
print(expected_df)

if p_value_ind < 0.05:
    print("\nReject H0: Gender and purchase behavior are associated")
else:
    print("\nFail to reject H0: No association between gender and purchase behavior")

In [ ]:
# Visualize contingency table
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

contingency_df.plot(kind='bar', stacked=True, ax=ax1, color=['green', 'red'], alpha=0.7)
ax1.set_title('Observed Frequencies')
ax1.set_ylabel('Count')
ax1.set_xlabel('Gender')
ax1.legend(title='Purchase Decision')

sns.heatmap(contingency_df, annot=True, fmt='d', cmap='YlOrRd', ax=ax2)
ax2.set_title('Contingency Table Heatmap')

plt.tight_layout()
plt.show()

## 8. The Unifying Principle

Despite the variety of techniques, the underlying logic remains identical:

Test Statistic = Observed Signal / Expected Random Variation

| Test | Signal | Random Variation | Distribution |
|------|--------|------------------|--------------|
| Two-sample t-test | xbar1 - xbar2 | sqrt(s1^2/n1 + s2^2/n2) | t-distribution |
| ANOVA | Between-group variance | Within-group variance | F-distribution |
| Two-proportion z-test | phat1 - phat2 | sqrt(phat(1-phat)(1/n1+1/n2)) | Normal |
| Chi-square | sum((O-E)^2/E) | Expected counts | chi-square |

**Key Question:** Is the observed pattern too extreme to plausibly attribute to random chance alone?

## 9. Practical Applications

These methods power real-world analytics:

- A/B Testing Systems - Comparing conversion rates
- Quality Control - Comparing defect rates across production lines
- Medical Research - Comparing treatment efficacy
- Market Research - Analyzing survey responses
- Manufacturing - Comparing process consistency
- Finance - Comparing portfolio performance
- Education - Comparing teaching methods
- Public Policy - Evaluating program effectiveness

## 10. Practice Exercises

### Exercise 1: Two-Sample t-Test
A company wants to compare the effectiveness of two training programs. Program A: 25 employees, average score 85, std dev 8. Program B: 25 employees, average score 90, std dev 7. Is Program B significantly better? Conduct a hypothesis test at alpha=0.05.

In [ ]:
# Exercise 1 Solution
n1, mean1, std1 = 25, 85, 8
n2, mean2, std2 = 25, 90, 7

se_ex = np.sqrt((std1**2/n1) + (std2**2/n2))
t_ex = (mean2 - mean1) / se_ex
df_ex = n1 + n2 - 2
p_ex = 2 * (1 - stats.t.cdf(abs(t_ex), df_ex))

print(f"t-statistic: {t_ex:.4f}")
print(f"p-value: {p_ex:.4f}")
print(f"Conclusion: {'Reject H0' if p_ex < 0.05 else 'Fail to reject H0'}")

### Exercise 2: Chi-Square Test of Independence
A survey asks 200 people about their device preference (Mobile/Desktop) and age group (Under 30/Over 30). Data: Under 30 Mobile=60, Desktop=40; Over 30 Mobile=30, Desktop=70. Is device preference associated with age group?

In [ ]:
# Exercise 2 Solution
survey_data = np.array([[60, 40], [30, 70]])
chi2_ex, p_ex, dof_ex, expected_ex = stats.chi2_contingency(survey_data)

print(f"Chi-square: {chi2_ex:.4f}")
print(f"p-value: {p_ex:.4f}")
print(f"Conclusion: {'Reject H0 - Association exists' if p_ex < 0.05 else 'Fail to reject H0 - No association'}")